<a href="https://colab.research.google.com/github/SrijaMadarapu8/AI-ML/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#What Are Transformers?

A transformer is a neural network architecture that processes sequences, such as text, using attention mechanisms to determine how different elements in the sequence relate to one another.

Transformers power many modern AI systems, including ChatGPT, Claude, Gemini, GitHub Copilot, and DALL-E.

**The Transformer Workflow**

A simplified workflow for processing text with a transformer looks like this:

Text → Tokenizer → Token IDs → Embeddings → Attention → Context-Aware Representations → Task Output

**Step 1: Tokenization (Text → Token IDs)**

Transformers don't read words directly, they read tokens. That's why the workflow starts with a tokenizer to convert text into token IDs the model can process.

For example, if we start with a sentence like "the cat is orange", the tokenizer:

1.   Breaks it into tokens (small pieces of text)
2.   Maps each token to a token ID (a number)

The exact tokens and IDs depend on the tokenizer used by the model. Let's try it out with the Qwen tokenizer:

In [ ]:
from transformers import AutoTokenizer

# Load a tokenizer (using a small Qwen3 model)
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Example sentence
text = "the cat is orange"

print(f"Using tokenizer from: {model_name}\n")
print("Original text:")
print(f"  '{text}'\n")

# Step 1: Break into tokens
tokens = tokenizer.tokenize(text)
print("Step 1 - Tokens (small pieces of text):")
print(f"  {tokens}\n")
print("  --> The 'Ġ' symbol represents a SPACE before the token")
print("      So 'Ġcat' means ' cat' (space + cat)\n")

# Step 2: Convert to token IDs
token_ids = tokenizer.encode(text)
print("Step 2 - Token IDs (numbers the model uses):")
print(f"  {token_ids}\n")

print("💡 The tokenizer converted 'the cat is orange' into numbers!")
print("   Now the transformer can process them.")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Using tokenizer from: Qwen/Qwen3-0.6B

Original text:
  'the cat is orange'

Step 1 - Tokens (small pieces of text):
  ['the', 'Ġcat', 'Ġis', 'Ġorange']

  --> The 'Ġ' symbol represents a SPACE before the token
      So 'Ġcat' means ' cat' (space + cat)

Step 2 - Token IDs (numbers the model uses):
  [1782, 8251, 374, 18575]

💡 The tokenizer converted 'the cat is orange' into numbers!
   Now the transformer can process them.


In [ ]:
# Compare tokenization of different sentences
sentences = [
    "the cat is orange",
    "the cat is tabby",
    "the cat is underweight"
]

print(f"Tokenization with {model_name}:\n")

for sentence in sentences:
    tokens = tokenizer.tokenize(sentence)
    token_ids = tokenizer.encode(sentence)

    print(f"Sentence: '{sentence}'")
    print(f"  Tokens: {tokens}")
    print(f"  Token IDs: {token_ids}\n")


Tokenization with Qwen/Qwen3-0.6B:

Sentence: 'the cat is orange'
  Tokens: ['the', 'Ġcat', 'Ġis', 'Ġorange']
  Token IDs: [1782, 8251, 374, 18575]

Sentence: 'the cat is tabby'
  Tokens: ['the', 'Ġcat', 'Ġis', 'Ġtab', 'by']
  Token IDs: [1782, 8251, 374, 5651, 1694]

Sentence: 'the cat is underweight'
  Tokens: ['the', 'Ġcat', 'Ġis', 'Ġunder', 'weight']
  Token IDs: [1782, 8251, 374, 1212, 4765]



**Step 2: Embeddings (Token IDs → Vectors)**

Next, the transformer takes those token IDs and converts each one into an embedding, which is a vector of numbers.

You can think of embeddings as coordinates in a multi-dimensional space. During training, the model learns to place tokens so that those used in similar contexts end up close together.

For example, tokens related to similar ideas may end up near each other in the embedding space:



*  Days of the week like "Monday", "Tuesday", and "Wednesday"
*  Countries like "France", "Germany", and "Spain"
*  Colors like "red", "blue", and "green"

Because these tokens often appear in similar contexts, their embeddings become similar.

Let's load a model and look at its embeddings:

In [ ]:
from transformers import AutoModel
import torch

# Load a modern transformer model (Qwen3 from 2024)
model_name = "Qwen/Qwen3-0.6B"
print(f"Loading {model_name}...")
model = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("✓ Model loaded!\n")

# Get the token IDs for our sentence
text = "the cat is orange"
token_ids = tokenizer.encode(text, return_tensors="pt")

print(f"Text: '{text}'\n")
print(f"Token IDs shape: {token_ids.shape}")
print(f"  → {token_ids.shape[1]} tokens\n")

# Get embeddings from the model's embedding layer
with torch.no_grad():
    # The embedding layer converts token IDs to vectors
    embedding_layer = model.get_input_embeddings()
    embeddings = embedding_layer(token_ids)

print(f"Embeddings shape: {embeddings.shape}")
print(f"  → {embeddings.shape[1]} tokens")
print(f"  → {embeddings.shape[2]} dimensions per token")
print(f"\n💡 Each token ID became a {embeddings.shape[2]}-dimensional vector!")

Loading Qwen/Qwen3-0.6B...


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded!

Text: 'the cat is orange'

Token IDs shape: torch.Size([1, 4])
  → 4 tokens

Embeddings shape: torch.Size([1, 4, 1024])
  → 4 tokens
  → 1024 dimensions per token

💡 Each token ID became a 1024-dimensional vector!


In [ ]:
import torch.nn.functional as F

# Get the embedding layer
embedding_layer = model.get_input_embeddings()

# Function to get embedding for a single word
def get_word_embedding(word):
    token_id = tokenizer.encode(word, add_special_tokens=False)[0]
    embedding = embedding_layer(torch.tensor([[token_id]]))
    return embedding[0, 0]

# Compare words from different categories
words_animals = ["cat", "dog"]
words_colors = ["red", "blue"]
words_countries = ["India", "Canada"]

all_words = words_animals + words_colors + words_countries
word_embeddings = {word: get_word_embedding(word) for word in all_words}

print("Let's compare words from DIFFERENT categories:\n")

# Compare cat (animal) to other words
cat_embedding = word_embeddings["cat"]

print("How similar are these words to 'cat' (an animal)?\n")
for word in all_words:
    if word == "cat":
        continue
    similarity = F.cosine_similarity(
        cat_embedding.unsqueeze(0),
        word_embeddings[word].unsqueeze(0)
    ).item()

    category = ""
    if word in words_animals:
        category = "(animal)"
    elif word in words_colors:
        category = "(color)"
    elif word in words_countries:
        category = "(country)"

    print(f"  cat ↔ {word:10s} {category:10s}: {similarity:.3f}")

print("\n💡:")
print("   • 'dog' (another animal) is the most similar to 'cat'")
print("   • Colors and countries are less similar to 'cat'")
print("   • Embeddings encode aspects of a token's meaning as a vector of numbers")

Let's compare words from DIFFERENT categories:

How similar are these words to 'cat' (an animal)?

  cat ↔ dog        (animal)  : 0.287
  cat ↔ red        (color)   : 0.083
  cat ↔ blue       (color)   : 0.053
  cat ↔ India      (country) : 0.008
  cat ↔ Canada     (country) : 0.057

💡:
   • 'dog' (another animal) is the most similar to 'cat'
   • Colors and countries are less similar to 'cat'
   • Embeddings encode aspects of a token's meaning as a vector of numbers


**Step 3: Attention**

Once each token has been converted into an embedding vector, we have a row of vectors. But at this point, each embedding only represents that token in isolation. The transformer needs a way to update these embeddings so each token understands its context (i.e. the words around it).

**What is Attention?**

Imagine reading the sentence: "The chicken didn't cross the road because it was too tired."

What does "it" refer to? You know it's "the chicken" because you connect it with earlier words in the sentence. Attention works in a similar way. It lets each word look at the other words in the sequence and decide which ones are most relevant for understanding it. By giving different levels of importance (or attention) to different words, the model can understand context and how a word’s meaning can change depending on the surrounding words.

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import torch.nn.functional as F

# Load model
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Compare the same word "bank" in different contexts
sentences = [
    "I went to the bank",        # bank = financial institution
    "I sat by the river bank"    # bank = edge of river
]

representations = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt")

    # Get contextual representation AFTER attention
    with torch.no_grad():
        outputs = model(**inputs)

    # Find "bank" token and get its vector
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    for i, token in enumerate(tokens):
        if "bank" in token.lower():
            bank_idx = i
            break
    representations.append(outputs.last_hidden_state[0, bank_idx])

    print(f"'{sentence}'")

# Compare the two "bank" vectors
similarity = F.cosine_similarity(
    representations[0].unsqueeze(0),
    representations[1].unsqueeze(0)
).item()

print(f"\nSimilarity between the two 'bank' vectors: {similarity:.3f}")
print("\n💡 Same word, different contexts → different vectors!")
print("   Attention lets each word look at other tokens in the sequence to understand its meaning.")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'I went to the bank'
'I sat by the river bank'

Similarity between the two 'bank' vectors: 0.727

💡 Same word, different contexts → different vectors!
   Attention lets each word look at other tokens in the sequence to understand its meaning.


**Causal Language Modeling**

In causal language modeling, we take the transformer's context-aware vectors and use them to predict what token should come next. For example, if we give the model a story starter like "Once upon a time", it predicts a probability distribution over possible next tokens and chooses one to continue the story.

It's called causal because the model can't look ahead. At each position, it predicts the next token using only the tokens that came before it. This left-to-right constraint, only seeing past tokens and never future ones, is what makes causal language models well suited for text generation.

In [ ]:
# Demonstrating one-token-at-a-time generation
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Start with our prompt
text = "The weather today is"
print(f"Starting text: '{text}'\n")

# Generate 5 tokens, showing the process
for step in range(5):
    # Tokenize current text
    inputs = tokenizer(text, return_tensors="pt")

    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)

    # Get predictions for the NEXT token
    next_token_logits = outputs.logits[0, -1, :]

    # Get the most likely next token
    next_token_id = next_token_logits.argmax()
    next_token = tokenizer.decode(next_token_id)

    # Add it to our text
    text += next_token

    print(f"Step {step + 1}: Added '{next_token}' → '{text}'")

print("\n💡 This is how modern language models generate text: one token at a time.")
print("   At each step, the model predicts the next token based on all previous tokens.")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Starting text: 'The weather today is'

Step 1: Added ' ' → 'The weather today is '
Step 2: Added '2' → 'The weather today is 2'
Step 3: Added '0' → 'The weather today is 20'
Step 4: Added ' degrees' → 'The weather today is 20 degrees'
Step 5: Added '.' → 'The weather today is 20 degrees.'

💡 This is how modern language models generate text: one token at a time.
   At each step, the model predicts the next token based on all previous tokens.


**Masked Language Modeling**

Another common language-modeling objective is masked language modeling (MLM).

Instead of predicting the next word, MLM hides a word in a sentence and trains the model to guess the missing word. The model can use both the left and the right side of the sentence to figure out the masked word.

Models trained this way are especially good at language understanding tasks where you want the model to interpret text using full context, such as text classification, labeling words in a sentence (like names, dates, places), and extracting information and answering questions based on a passage.

In [ ]:
from transformers import pipeline

# Load a fill-mask model (masked language model)
# Note: We use BERT here because Qwen is a causal model and can't do masked prediction
print("Loading masked language model (BERT)...")
fill_mask = pipeline("fill-mask", model="bert-base-uncased")
print("✓ Model loaded!\n")

# Use sentences with strong contextual clues
sentences = [
    "The sky is [MASK].",
    "I drink [MASK] in the morning."
]

for sentence in sentences:
    print(f"Sentence: '{sentence}'")
    print("Top predictions:\n")

    results = fill_mask(sentence, top_k=3)

    for i, result in enumerate(results, 1):
        word = result['token_str'].strip()
        score = result['score']
        filled = result['sequence']
        print(f"  {i}. '{word}' ({score:.1%})")
        print(f"      → {filled}")

    print("\n" + "="*60 + "\n")

print("💡 Masked Language Modeling uses BOTH left AND right context, it sees the FULL sentence.")
print("   This makes it great for understanding.")

Loading masked language model (BERT)...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

✓ Model loaded!

Sentence: 'The sky is [MASK].'
Top predictions:

  1. 'clear' (20.1%)
      → the sky is clear.
  2. 'blue' (19.0%)
      → the sky is blue.
  3. 'dark' (13.3%)
      → the sky is dark.


Sentence: 'I drink [MASK] in the morning.'
Top predictions:

  1. 'it' (31.5%)
      → i drink it in the morning.
  2. 'coffee' (14.6%)
      → i drink coffee in the morning.
  3. 'beer' (4.8%)
      → i drink beer in the morning.


💡 Masked Language Modeling uses BOTH left AND right context, it sees the FULL sentence.
   This makes it great for understanding.
